# EDA theo Dashboard hiện tại

Notebook đọc các bảng do `scripts/clean_data.py` tạo và minh họa các biểu đồ đang dùng trong Dashboard. Notebook chỉ đọc và hiển thị, không ghi file.

## 1. Tổng quan bán hàng

Dùng bảng `orders_enriched.csv` để xem KPI và cơ cấu doanh thu theo danh mục.

In [6]:
from pathlib import Path
import pandas as pd
import plotly.express as px

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'data' / 'processed' / 'orders_enriched.csv').is_file())
P = ROOT / 'data' / 'processed'
orders = pd.read_csv(P / 'orders_enriched.csv', parse_dates=['Order Date'], low_memory=False)
orders.head()

,Order ID,Order Date,Customer ID,Country,Region,Category,Sub-Category,Sales,Quantity,Profit,Data Source,Manager
0,AG-2012-TB112803-40909,2012-01-01,TB-112803,Algeria,North Africa,Office Supplies,Storage,408.300,2,106.140,Global Superstore (2012-2015),Lindiwe Afolayan
1,IN-2012-JH159857-40909,2012-01-01,JH-159857,Australia,Oceania,Office Supplies,Supplies,120.366,3,36.036,Global Superstore (2012-2015),Kauri Anaru
2,IN-2012-JH159857-40909,2012-01-01,JH-159857,Australia,Oceania,Furniture,Furnishings,113.670,5,37.770,Global Superstore (2012-2015),Kauri Anaru
3,IT-2012-EM14140124-40909,2012-01-01,EM-14140124,Sweden,Northern Europe,Office Supplies,Paper,44.865,3,-26.055,Global Superstore (2012-2015),Miina Nylund
4,IN-2012-JH159857-40909,2012-01-01,JH-159857,Australia,Oceania,Office Supplies,Paper,55.242,2,15.342,Global Superstore (2012-2015),Kauri Anaru


In [7]:
category_sales = orders.groupby('Category', as_index=False)['Sales'].sum().sort_values('Sales', ascending=False)
px.bar(category_sales, x='Category', y='Sales', title='Tổng quan — Doanh thu theo danh mục')

In [8]:
monthly = orders.assign(Month=orders['Order Date'].dt.to_period('M').dt.to_timestamp()).groupby('Month', as_index=False)['Sales'].sum()
px.line(monthly, x='Month', y='Sales', markers=True, title='Tổng quan — Xu hướng doanh thu theo tháng')

## 2. Phân tích địa lý

Nhóm biểu đồ này trả lời câu hỏi doanh thu đang tập trung ở quốc gia hoặc khu vực nào.

In [9]:
country_sales = orders.groupby('Country', as_index=False)['Sales'].sum().sort_values('Sales', ascending=False).head(15)
px.bar(country_sales, x='Sales', y='Country', orientation='h', title='Địa lý — 15 quốc gia có doanh thu cao nhất')

In [10]:
region_category = orders.pivot_table(index='Region', columns='Category', values='Sales', aggfunc='sum', fill_value=0)
px.imshow(region_category, text_auto='.2s', aspect='auto', title='Địa lý — Doanh thu theo khu vực và danh mục')

## 3. Phân khúc RFM

RFM gồm lần mua gần nhất, số lần mua và số tiền đã chi. Scatter plot giúp nhìn mối quan hệ giữa tần suất mua và giá trị khách hàng.

In [11]:
rfm = pd.read_csv(P / 'rfm_customers.csv', low_memory=False)
rfm[['Customer ID', 'Recency', 'Frequency', 'Monetary', 'Segment']].head()

,Customer ID,Recency,Frequency,Monetary,Segment
0,AA-10315,2377,5,5563.56,Champions
1,AA-10315,185,5,5563.56,Champions
2,AA-10315102,3646,2,544.66,Loyal Customers
3,AA-10315120,4247,1,2713.41,Need Attention
4,AA-10315139,3437,4,2955.80,Champions


In [12]:
segment_counts = rfm['Segment'].value_counts().rename_axis('Segment').reset_index(name='Customers')
px.pie(segment_counts, names='Segment', values='Customers', hole=.45, title='RFM — Tỷ trọng các phân khúc khách hàng')

In [13]:
px.scatter(rfm.sample(min(5000, len(rfm)), random_state=1), x='Frequency', y='Monetary', color='Segment', hover_data=['Country'], title='RFM — Frequency và Monetary')

## 4. Dự báo doanh thu

Bảng `monthly_sales.csv` là dữ liệu tổng hợp theo tháng, dùng làm đầu vào cho trang Dự báo. Dashboard sau đó hiển thị các kịch bản 3 năm, 5 năm và 10 năm.

In [14]:
monthly_sales = pd.read_csv(P / 'monthly_sales.csv', parse_dates=['Month'])
monthly_sales.head()

,Data Source,Month,Sales,Profit,Orders,Customers
0,Global Superstore (2012-2015),2012-01-01,98898.48886,8321.80096,216,215
1,Global Superstore (2012-2015),2012-02-01,103717.92328,14323.06088,203,202
2,Global Superstore (2012-2015),2012-03-01,135746.40206,13089.62696,266,266
3,Global Superstore (2012-2015),2012-04-01,114051.59118,13223.17058,258,255
4,Global Superstore (2012-2015),2012-05-01,158212.93880,14641.56230,317,314


In [15]:
forecast_history = monthly_sales.groupby('Month', as_index=False)['Sales'].sum()
px.line(forecast_history, x='Month', y='Sales', title='Dự báo — Doanh thu lịch sử theo tháng')

## 5. Walmart và BigMart

Hai bảng này phục vụ hai trang chuyên đề riêng, không gộp vào bảng bán hàng chính.

In [16]:
walmart = pd.read_csv(P / 'walmart_weekly_enriched.csv', nrows=100000, parse_dates=['Date'], low_memory=False)
px.scatter(walmart.sample(min(5000, len(walmart)), random_state=1), x='Size', y='Weekly_Sales', color='Type', title='Walmart — Quy mô cửa hàng và doanh số tuần')

In [17]:
bigmart = pd.read_csv(P / 'bigmart_cleaned.csv', low_memory=False)
px.scatter(bigmart, x='Item_Visibility', y='Item_Outlet_Sales', color='Outlet_Type', title='BigMart — Mức độ trưng bày và doanh số')